# Analyzing results and saving data

This file can be used to see the results and plots from a specific run, and to save the results in csv files for final plotting through "analysis_all_results.ipynb".

Edit the settings to decide which scenario to run.

In [ ]:
import logging
import warnings
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.WARNING)


In [ ]:
# CASE SETTINGS
case = 'BASE'
case_path = 'base'
code = 'B_'

In [ ]:
# SENSITIVITY ANALYSES SETTINGS
discountrate = 0.04
cost_loop = False
gas_price_4 = False
gas_price_1 = False

In [ ]:
# SAVE SETTINGS
save_to_csv = False

In [ ]:
# LOAD NETWORKS
path_name = '../' + case_path + '/' + code
years = [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040]
networks = [pypsa.Network(path_name + str(year) + '.nc') for year in years]
networks = dict(zip(years, networks))
first_year = years[0]
final_year = years[-1]

In [ ]:
# RETREIVING COSTS FOR COST SENSITIVITY ANALYSIS
# Dictionary for cost_year, 2024 has cost year 2020, 2025 - 2029 has cost year 2025, 2030 - 2034 has cost year 2030, and 2035 - 2040 has cost year 2035
cost_years = {
    2024: '2020',
    2025: '2025',
    2026: '2025',
    2027: '2025',
    2028: '2025',
    2029: '2025',
    2030: '2030',
    2031: '2030',
    2032: '2030',
    2033: '2030',
    2034: '2030',
    2035: '2035',
    2036: '2035',
    2037: '2035',
    2038: '2035',
    2039: '2035',
    2040: '2035',
}   


scale_cost_sudden = {
    ('OCGT', 2025): 1.0,
    ('OCGT', 2026): 1.0,
    ('OCGT', 2027): 0.20,
    ('OCGT', 2028): 0.20,
    ('OCGT', 2029): 0.20,
    ('OCGT', 2030): 0.20,
    ('OCGT', 2031): 0.20,
    ('OCGT', 2032): 0.20,
    ('OCGT', 2033): 0.20,
    ('OCGT', 2034): 0.20,
    ('OCGT', 2035): 0.20,
    ('OCGT', 2036): 0.20,
    ('OCGT', 2037): 0.20,
    ('OCGT', 2038): 0.20,
    ('OCGT', 2039): 0.20,
    ('OCGT', 2040): 0.20,

    ('CCGT', 2025): 1.0,
    ('CCGT', 2026): 1.0,
    ('CCGT', 2027): 0.20,
    ('CCGT', 2028): 0.20,
    ('CCGT', 2029): 0.20,
    ('CCGT', 2030): 0.20,
    ('CCGT', 2031): 0.20,
    ('CCGT', 2032): 0.20,
    ('CCGT', 2033): 0.20,
    ('CCGT', 2034): 0.20,
    ('CCGT', 2035): 0.20,
    ('CCGT', 2036): 0.20,
    ('CCGT', 2037): 0.20,
    ('CCGT', 2038): 0.20,
    ('CCGT', 2039): 0.20,
    ('CCGT', 2040): 0.20,

    ('oil', 2025): 1.0,
    ('oil', 2026): 1.0,
    ('oil', 2027): 0.29,
    ('oil', 2028): 0.29,
    ('oil', 2029): 0.29,
    ('oil', 2030): 0.29,
    ('oil', 2031): 0.29,
    ('oil', 2032): 0.29,
    ('oil', 2033): 0.29,
    ('oil', 2034): 0.29,
    ('oil', 2035): 0.29,
    ('oil', 2036): 0.29,
    ('oil', 2037): 0.29,
    ('oil', 2038): 0.29,
    ('oil', 2039): 0.29,
    ('oil', 2040): 0.29,
}

scale_cost_gradual = {
    ('OCGT', 2025): 0.79,
    ('OCGT', 2026): 0.65,
    ('OCGT', 2027): 0.55,
    ('OCGT', 2028): 0.48,
    ('OCGT', 2029): 0.43,
    ('OCGT', 2030): 0.38,
    ('OCGT', 2031): 0.35,
    ('OCGT', 2032): 0.32,
    ('OCGT', 2033): 0.29,
    ('OCGT', 2034): 0.27,
    ('OCGT', 2035): 0.25,
    ('OCGT', 2036): 0.24,
    ('OCGT', 2037): 0.22,
    ('OCGT', 2038): 0.21,
    ('OCGT', 2039): 0.20,
    ('OCGT', 2040): 0.20,

    ('CCGT', 2025): 0.79,
    ('CCGT', 2026): 0.65,
    ('CCGT', 2027): 0.55,
    ('CCGT', 2028): 0.48,
    ('CCGT', 2029): 0.43,
    ('CCGT', 2030): 0.38,
    ('CCGT', 2031): 0.35,
    ('CCGT', 2032): 0.32,
    ('CCGT', 2033): 0.29,
    ('CCGT', 2034): 0.27,
    ('CCGT', 2035): 0.25,
    ('CCGT', 2036): 0.24,
    ('CCGT', 2037): 0.22,
    ('CCGT', 2038): 0.21,
    ('CCGT', 2039): 0.20,
    ('CCGT', 2040): 0.20,

    ('oil', 2025): 0.86,
    ('oil', 2026): 0.75,
    ('oil', 2027): 0.67,
    ('oil', 2028): 0.60,
    ('oil', 2029): 0.55,
    ('oil', 2030): 0.50,
    ('oil', 2031): 0.46,
    ('oil', 2032): 0.43,
    ('oil', 2033): 0.40,
    ('oil', 2034): 0.37,
    ('oil', 2035): 0.35,
    ('oil', 2036): 0.33,
    ('oil', 2037): 0.32,
    ('oil', 2038): 0.30,
    ('oil', 2039): 0.29,
    ('oil', 2040): 0.29,
}


In [ ]:
# Colors
red1 = '#891D2D'
red2 = '#BA3B31'
orange = '#F58221'
yellow = '#FCAF19'
brown = '#440A15'
brown2 = '#B45419'
purple1 = '#3B1053'
purple2 = '#76518E'
purple3 = '#B69DC7'
teal1 = '#032838'
teal2 = '#154655'
teal3 = '#527D77'
teal4 = '#8DB5AF'
teal1 = '#294839'
green1 = '#6DA08C'
green2 = '#6E966E'
green3 = '#A3BDA3'
beige1 = '#7A693B'
beige2 = '#A89677'
beige3 = '#D2CDAD'
grey1 = '#E7E7E7'
grey2 = '#D7D7D7'
grey3 = '#C6C6C6'
grey4 = '#939393'
blue1 = '#3EA1C0'

In [ ]:
# Helper functions

def scale_costs(year, case):
    """
    Scales the costs for OCGT, CCGT and oil based on the case (sudden or gradual).  
    Parameters:
    cost_year (str): The year for which the costs are being scaled.
    case (str): The case for which the costs are being scaled ('Sudden' or 'Gradual', or 'sudden' or 'gradual).
    Returns:
    dict: A dictionary with the scaled costs for OCGT, CCGT and oil.
    """
    scale_costs = {}
    if case.lower() == 'sudden':
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = scale_cost_sudden[(tech, year)]
    elif case.lower() == 'gradual':
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = scale_cost_gradual[(tech, year)]
    else: # Nothing to scale if case is not sudden or gradual
        for tech in ['OCGT', 'CCGT', 'oil']:
            for year in range(2020, 2041):
                scale_costs[(tech, year)] = 1.0
    return scale_costs  

def get_power_prod(year):
    network = networks[year]
    carrier_list = network.generators.carrier.unique()
    carrier_list_2 = network.storage_units.carrier.unique()
    carriers = np.concatenate((carrier_list, carrier_list_2), axis=None)
    production_data = {}
    first_date = "2013-01-01"
    second_date = "2013-12-31"
 
    production_data["CCGT"] = get_snapshot_generation(year, first_date, second_date, "CCGT").sum() /1e6
    production_data["OCGT"] = get_snapshot_generation(year, first_date, second_date, "OCGT").sum() /1e6
    production_data["oil"] = get_snapshot_generation(year, first_date, second_date, "oil").sum() /1e6
    production_data["geothermal"] = get_snapshot_generation(year, first_date, second_date, "geothermal").sum() /1e6
    production_data["hydro"] = get_snapshot_generation(year, first_date, second_date, "hydro").sum() /1e6
    production_data["onwind"] = get_snapshot_generation(year, first_date, second_date, "onwind").sum() /1e6
    production_data["solar"] = get_snapshot_generation(year, first_date, second_date, "solar").sum() /1e6
    production_data["biomass"] = get_snapshot_generation(year, first_date, second_date, "biomass").sum() /1e6
    production_data["ror"] = get_snapshot_generation(year, first_date, second_date, "ror").sum() /1e6
 
    df = pd.DataFrame([production_data]) # in TWh
    df['hydro'] += df.pop('ror')
    return df

def get_power_prod_hydro(year):
    #print(year)
    network = networks[year]
    carrier_list = network.generators.carrier.unique()
    carrier_list_2 = network.storage_units.carrier.unique()
    carriers = np.concatenate((carrier_list, carrier_list_2), axis=None)
    production_data = {}
    first_date = "2013-01-01"
    second_date = "2013-12-31"
    # for carrier in carriers:
    #     production_data[carrier] = get_snapshot_generation(year, first_date, second_date, carrier).sum() /1e6
 
    production_data["CCGT"] = get_snapshot_generation(year, first_date, second_date, "CCGT").sum() /1e6
    production_data["OCGT"] = get_snapshot_generation(year, first_date, second_date, "OCGT").sum() /1e6
    production_data["oil"] = get_snapshot_generation(year, first_date, second_date, "oil").sum() /1e6
    production_data["geothermal"] = get_snapshot_generation(year, first_date, second_date, "geothermal").sum() /1e6
    production_data["hydro"] = get_snapshot_generation(year, first_date, second_date, "hydro").sum() /1e6
    production_data["onwind"] = get_snapshot_generation(year, first_date, second_date, "onwind").sum() /1e6
    production_data["solar"] = get_snapshot_generation(year, first_date, second_date, "solar").sum() /1e6
    production_data["biomass"] = get_snapshot_generation(year, first_date, second_date, "biomass").sum() /1e6
    production_data["ror"] = get_snapshot_generation(year, first_date, second_date, "ror").sum() /1e6
    #production_data["load"] = get_snapshot_generation(year, first_date, second_date, "load").sum() /1e6
 
    df = pd.DataFrame([production_data]) # in TWh
    return df

def get_demand(year): 
    network = networks[year]
    el_demand =network.loads_t.p_set
    regional_demand = pd.DataFrame(el_demand.sum()/1e6)
    return regional_demand.sum()

def total_production(year):
    prod = pd.DataFrame(get_power_prod(year))
    #prod.drop('load', axis=1, inplace=True)
    return prod.sum().sum()

def get_power_mix(year):
    total_prod = total_production(year)
    prod = get_power_prod(year)
    prod_series = prod.iloc[0]#.drop('load', errors='ignore')
    fractions = prod_series / total_prod

    df = pd.DataFrame(fractions).transpose()
    df.index = [year]  
    return df

def custom_autopct(pct):
    return ('%1.1f%%' % pct) if pct > 0 else ''

def get_emissions(year):
    network = networks[year]
    emissions = network.generators_t.p / network.generators.efficiency * network.generators.carrier.map(network.carriers.co2_emissions)
    return emissions.sum().sum() / 1000000

def get_installed_capacity(year):
    network = networks[year]

    capacities = network.generators.groupby(by='carrier')['p_nom_opt'].sum()

    if 'ror' in capacities:
        capacities['hydro'] = capacities.get('hydro', 0) + capacities.pop('ror')

    if 'hydro' in network.storage_units.carrier.unique():
        hydro_capacity = network.storage_units[network.storage_units.carrier == 'hydro']['p_nom_opt'].sum()
        capacities['hydro'] += hydro_capacity

    capacities.pop('load')
    capacities_df = capacities.to_frame().transpose()
    capacities_df.index = [year]

    return capacities_df

def get_installed_capacity_charge(year):
    network = networks[year]
    capacities_discharge = network.links.groupby(by='carrier')['p_nom'].sum()
    capacities_discharge_df = capacities_discharge.to_frame().transpose()
    capacities_discharge_df.index = [year]
    return capacities_discharge_df

def get_installed_capacity_battery(year):
    network = networks[year]
    storage_capacity = network.stores.groupby(by='carrier')['e_nom'].sum()
    storage_capacity_df = storage_capacity.to_frame().transpose()
    storage_capacity_df.index = [year]
    return storage_capacity_df

def get_installed_capacity_lines(year):
    network = networks[year]
    lines_capacity = network.lines.groupby(by='carrier')['s_nom'].sum()
    lines_capacity_df = lines_capacity.to_frame().transpose()
    lines_capacity_df.index = [year]
    return lines_capacity_df

def get_snapshot_generation(year, first_date, second_date, carrier):
    network = networks[year]
    if carrier == 'hydro':
        generation = network.storage_units_t.p_dispatch[first_date:second_date].groupby(network.storage_units.carrier, axis=1).sum()[carrier]
    elif carrier == 'battery':
        generation = network.stores_t.p.loc[first_date:second_date].groupby(network.stores.carrier, axis=1).sum()[carrier]
    else:
        generation = network.generators_t.p.loc[first_date:second_date].groupby(network.generators.carrier, axis=1).sum()[carrier]
    return generation

def get_snapshot_demand(year, first_date, second_date):
    network = networks[year]
    demand = network.loads_t.p_set.loc[first_date:second_date].sum(axis=1)*-1
    return demand

def get_new_installed(years, merge_ror=True):

    data_list = []

    for y in years:
        net = networks[y]

        capacity = net.generators[["p_nom_opt","carrier","p_nom"]]
        hydro = net.storage_units[["p_nom_opt","carrier","p_nom"]]

        caps = pd.concat([capacity, hydro], ignore_index=True)

        caps["p_change"] = caps["p_nom_opt"] - caps["p_nom"]
        caps["year"] = y

        data_list.append(caps[["year","carrier","p_change"]])

    data_agg = pd.concat(data_list)

    grouped = data_agg.groupby(['year','carrier']).sum().unstack()

    grouped.columns = grouped.columns.droplevel(0)
    grouped = grouped.clip(lower=0)
    grouped = grouped.drop('load', axis=1)

    if merge_ror and "ror" in grouped.columns:
        grouped['hydro'] += grouped.pop('ror')

    return grouped

def get_new_installed_battery(years):
    capacity = {'Charger':[], 'Discharger':[],'Battery Storage':[], 'year':[]}

    for y in years:
        net=networks[y]

        capacity["year"].append(y)

        charger_capacity = net.links.groupby('carrier').p_nom.sum().get('battery charger', 0)
        charger_next_capacity = net.links.groupby('carrier').p_nom_opt.sum().get('battery charger', 0)
        capacity['Charger'].append(charger_next_capacity-charger_capacity)

        discharger_capacity = net.links.groupby('carrier').p_nom.sum().get('battery discharger', 0)
        discharger_next_capacity = net.links.groupby('carrier').p_nom_opt.sum().get('battery discharger', 0)
        capacity['Discharger'].append(discharger_next_capacity-discharger_capacity)

        battery_storage_capacity = net.stores.groupby('carrier').e_nom.sum().get('battery', 0)
        battery_storage_next_capacity = net.stores.groupby('carrier').e_nom_opt.sum().get('battery', 0)
        capacity['Battery Storage'].append(battery_storage_next_capacity-battery_storage_capacity)

    capacity_battery_df = pd.DataFrame(capacity)
    capacity_battery_df.set_index("year", inplace=True)

    return capacity_battery_df

def get_new_installed_lines(years):
    data_agg = pd.DataFrame({})

    for y in years:
        net = networks[y]
        lines = pd.DataFrame(net.lines)

        lines["line_id"] = lines.index
        
        lines["p_change"] = lines["s_nom_opt"] - lines["s_nom"]
        lines["year"] = np.ones(len(lines["s_nom_opt"]), dtype=int) * y

        data_agg = pd.concat([data_agg, lines[["year", "line_id", "p_change"]]])

    grouped_cap_change_L = data_agg.groupby(['year', 'line_id']).sum().unstack()

    grouped_cap_change_L.columns = grouped_cap_change_L.columns.droplevel(0)
    grouped_cap_change_L = grouped_cap_change_L.clip(lower=0)
    grouped_cap_change_L = grouped_cap_change_L.sort_index(axis=1)

    return grouped_cap_change_L

def rename_columns(df):
    new_names = ['Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Geothermal', 'Battery']
    old_names = ['biomass', 'oil', 'onwind', 'solar', 'hydro', 'geothermal', 'battery']
    name_map= dict(zip(old_names, new_names))
    df = df.rename(columns=name_map)
    return df

def get_colors(carriers):
    colors = [beige2, beige3, teal3, beige1, teal4, yellow, teal2, brown, brown2]
    names = ['CCGT',    'OCGT',  'Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Battery', 'Geothermal']
    color_dict = dict(zip(names, colors))
    colors_new = [color_dict[carrier] for carrier in carriers]
    return colors_new

def get_marginal_cost(y, carrier):
    network = networks[y]
    carrier_data = network.generators.loc[network.generators['carrier'] == carrier]

    if not carrier_data.empty:
        marginal_cost = carrier_data['marginal_cost'].iloc[0]
        return marginal_cost
    else:
        return 0 

def get_subsidies():
    subsidies = {}

    if cost_loop:
        cost_timeline = [
            (2020, {'CCGT': 47.82, 'OCGT': 65.19, 'oil': 157.35}),
            (2025, {'CCGT': 46.97, 'OCGT': 64.45, 'oil': 157.37}),
            (2030, {'CCGT': 46.13, 'OCGT': 63.73, 'oil': 157.37}),
            (2035, {'CCGT': 45.72, 'OCGT': 63.02, 'oil': 157.36}),
            (2040, {'CCGT': 45.32, 'OCGT': 62.32, 'oil': 157.36}),
        ]
    elif gas_price_1:
        cost_timeline = [
            (2020, {'CCGT': 44.88, 'OCGT': 55.56, 'oil': 128.22}),
            (2030, {'CCGT': 46.74, 'OCGT': 57.94, 'oil': 128.22}),
        ]
    elif gas_price_4:
        cost_timeline = [
            (2020, {'CCGT': 71.15, 'OCGT': 89.34, 'oil': 128.22}),
            (2028, {'CCGT': 76.32, 'OCGT': 95.98, 'oil': 128.22}),
            (2030, {'CCGT': 89.46, 'OCGT': 112.88, 'oil': 128.22}),
            (2035, {'CCGT': 98.01, 'OCGT': 123.87, 'oil': 128.22}),
        ]
    else:
        actual_cost = {'CCGT': 51.01, 'OCGT': 63.44, 'oil': 128.22}
    for y in years:
        if cost_loop or gas_price_1 or gas_price_4:
            for interval_start, costs in reversed(cost_timeline):
                if y >= interval_start:
                    actual_cost = costs
                    break
        for carrier in ['CCGT', 'OCGT', 'oil']:
            marginal_cost= get_marginal_cost(y, carrier)
            subsidies[(y, carrier)] = actual_cost[carrier] - marginal_cost
    return subsidies

def calculate_present_value(future_value, year, base_year, discount_rate=discountrate):
    return future_value / ((1 + discount_rate) ** (year - base_year))


def capacity_factors(year):
    carrier_mapping = {
        'Biomass': 'biomass',
        'Combined-Cycle Gas': 'CCGT',
        'Oil': 'oil',
        'Onshore Wind': 'onwind',
        'Open-Cycle Gas': 'OCGT',
        'Solar': 'solar',
        'Run of River': 'hydro',
        'Reservoir & Dam': 'hydro_store',
        'Geothermal': 'geothermal'
    }

    network = networks[year]
    network_stats = network.statistics()
    cp = {output: [] for output in carrier_mapping.values()}

    for carrier_stat, carrier in carrier_mapping.items():
        if carrier_stat == 'Reservoir & Dam':
            value = network_stats.loc['StorageUnit']['Capacity Factor'][carrier_stat]
            cp[carrier] = value
        else:
            value = network_stats.loc['Generator']['Capacity Factor'].drop('load')[carrier_stat]
            cp[carrier] = value

    return cp

def capcost_lines(network):

    capital_cost_df = pd.DataFrame()
    for line in network.lines.index:
        capital_cost = network.lines.loc[line, 'capital_cost']
        new_row = pd.DataFrame({'capital_cost': capital_cost}, index=[line])
        capital_cost_df = pd.concat([capital_cost_df, new_row])

    return capital_cost_df

def get_operational_costs(years):
    operational_costs_by_year = []

    for year in years:
        power_prod_df = get_power_prod_hydro(year)
        total_operational_cost = 0
        network = networks[year]
        
        for carrier in power_prod_df.columns:
            if carrier == 'hydro': # hydro is a storage unit, not a generator
                production = power_prod_df[carrier].iloc[0]
                operational_cost = network.storage_units.loc[network.storage_units['carrier'] == carrier, 'marginal_cost'].mean() * production
                total_operational_cost += operational_cost                
            
            elif carrier != 'load':  # Skip 'load' as it is not a generation carrier
                production = power_prod_df[carrier].iloc[0]
                operational_cost = network.generators.loc[network.generators['carrier'] == carrier, 'marginal_cost'].mean() * production
                total_operational_cost += operational_cost
        
        operational_costs_by_year.append(total_operational_cost)
    
    return operational_costs_by_year


def get_capital_costs(network):
    capital_cost_df = pd.DataFrame()
    for carrier in network.generators['carrier'].unique():
        if carrier != 'load':
            capital_cost = network.generators.loc[network.generators['carrier'] == carrier, 'capital_cost'].mean()
            new_row = pd.DataFrame({'capital_cost': capital_cost}, index=[carrier])
            capital_cost_df = pd.concat([capital_cost_df, new_row])
    # Add hydro costs
    
    hydro_cost = 270940.71528
    new_row = pd.DataFrame({'capital_cost': hydro_cost}, index=['hydro'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    # Add battery, charger, and discharger costs
    battery_cost = network.stores.loc[network.stores['carrier'] == 'battery', 'capital_cost'].mean()
    charger_cost = network.links.loc[network.links['carrier'] == 'battery charger', 'capital_cost'].mean()
    discharger_cost = network.links.loc[network.links['carrier'] == 'battery discharger', 'capital_cost'].mean()
    new_row = pd.DataFrame({'capital_cost': battery_cost}, index=['battery'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    new_row = pd.DataFrame({'capital_cost': charger_cost}, index=['battery charger'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    new_row = pd.DataFrame({'capital_cost': discharger_cost}, index=['battery discharger'])
    capital_cost_df = pd.concat([capital_cost_df, new_row])
    
    return capital_cost_df


### Network and demand plots

In [ ]:
# PLOT NETWORK OVERIEW 
networks[final_year].plot()

In [ ]:
# PLOT NETWORK WITH DEMAND AND LINE CAPACITY

max_node_size = 0.1  # maximum size of a node for plotting purposes [used in plots]

def normalize_node_size(values, max_node_size=max_node_size):
    return values / values.max() * max_node_size

fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw={
        "projection": ccrs.PlateCarree()
    },  
)

# Choose year to plot:
n = networks[first_year]

n.plot(
    margin=0.1,
    ax=None,
    geomap=True,  
    projection=None,
    bus_colors="cadetblue",
    bus_alpha=1,
    bus_sizes=normalize_node_size(
        n.loads_t.p_set.sum().rename("load").rename_axis("bus")
    ),
    bus_cmap=None,
    line_colors="rosybrown",
    link_colors="darkseagreen",  
    transformer_colors="orange",
    line_widths=n.lines.s_nom_opt / 3e2,
    link_widths=1.5,
    transformer_widths=1.5,
    line_cmap=None,
    link_cmap=None,  
    transformer_cmap=None,
    flow=None,
    branch_components=None,
    layouter=None,
    title=f"Network with demand and line capacity {first_year}",
    boundaries=None,
    geometry=False,
    jitter=None,
    color_geomap=True,
)  # None or True

ax.axis("off")
plt.show()

In [ ]:
# PLOT DEMAND DEVELOPMENT

total_demand = [get_demand(year) for year in years]
plt.figure(figsize=(10, 3))
plt.plot(years, total_demand, color=teal3)
plt.xticks(years, [int(year) for year in years])
plt.title('Total Demand Development')
plt.xlabel('Year')
plt.ylabel('Total Demand (TWh)')
plt.xlim(first_year,final_year)
plt.ylim(0,22)
plt.grid(axis='y')
plt.show()

### Cost calculation and plots

In [ ]:
# COST CALCULATIONS

new_installed_cap = get_new_installed(years, merge_ror=False)
new_installed_bat = get_new_installed_battery(years)
new_installed_lines = get_new_installed_lines(years)

capital_costs_final = []
generator_costs_final = []
line_costs_final = []
accumulated_new_cap = []
operational_costs_by_year = []
yearly_installed_lines = {}

for y in years:        
    
    network = networks[y]
    total_capex = 0

    capital_cost_df = get_capital_costs(network)
    capital_cost_lines = capcost_lines(network)

    total_gen_capex = 0
    total_bat_capex = 0
    total_line_capex = 0

    for carrier in new_installed_cap.columns:
        capex_value = new_installed_cap.loc[: y, carrier].sum() * capital_cost_df.loc[carrier, 'capital_cost']
        total_capex += capex_value
        total_gen_capex += capex_value


    bat_capex = new_installed_bat.loc[:y, 'Battery Storage'].sum() * capital_cost_df.loc['battery', 'capital_cost']
    bat_capex += new_installed_bat.loc[:y, 'Charger'].sum() * capital_cost_df.loc['battery charger', 'capital_cost']
    bat_capex += new_installed_bat.loc[:y, 'Discharger'].sum() * capital_cost_df.loc['battery discharger', 'capital_cost']
    total_bat_capex += bat_capex
    total_capex += bat_capex

    for line in new_installed_lines.columns:
        line_capex = new_installed_lines.loc[:y, str(line)].sum() * capital_cost_lines.loc[line, 'capital_cost']
        total_capex += line_capex
        total_line_capex += line_capex

    total_line_capacity = new_installed_lines.loc[y, :].sum()
    yearly_installed_lines[y] = total_line_capacity
    

    capital_costs_final.append(total_capex/1e6) # Convert to million EUR
    generator_costs_final.append((total_gen_capex + total_bat_capex)/1e6) # million EUR
    line_costs_final.append(total_line_capex/1e6) # million EUR

operational_costs_by_year = get_operational_costs(years) # million EUR

pv_gen_capex = [calculate_present_value(i, year, years[0], discountrate) for i, year in zip(generator_costs_final, years)]
pv_line_capex = [calculate_present_value(i, year, years[0], discountrate) for i, year in zip(line_costs_final, years)]
print('PyPSA gen cap cost: ', sum(pv_gen_capex))
print('PyPSA line cap cost: ', sum(pv_line_capex))
    

pv_capital_costs = [calculate_present_value(i, year, years[0], discountrate) for i, year in zip(capital_costs_final, years)]
pv_operational_costs = [calculate_present_value(i, year, years[0], discountrate) for i, year in zip(operational_costs_by_year, years)]

print('PyPSA total cap cost: ', sum(pv_capital_costs))
print('PyPSA total op cost: ', sum(pv_operational_costs))
print('PyPSA total: ',sum(pv_capital_costs) + sum(pv_operational_costs))

if save_to_csv:
    try:
        capcost_df = pd.read_csv('../result_data/capcost.csv')
        opex_df = pd.read_csv('../result_data/opex.csv')
    except FileNotFoundError:
        capcost_df = pd.DataFrame()
        opex_df = pd.DataFrame()
    
    capcost_df[case_path] = pv_capital_costs
    opex_df[case_path] = pv_operational_costs

    capcost_df.to_csv('../result_data/capcost.csv', index=False)
    opex_df.to_csv('../result_data/opex.csv', index=False)

    df_total_capacity = pd.DataFrame.from_dict(yearly_installed_lines, orient='index', columns=['Total_capacity'])
    df_total_capacity.to_csv('../result_data/' + case_path + '_new_line_capacity.csv')

In [ ]:
# PLOT PV PER YEAR 

total_pv = [pv_capital_costs[i] + pv_operational_costs[i] for i in range(len(years))]
total_pv_df = pd.DataFrame(total_pv)

print('Capital costs: ', sum(pv_capital_costs)/1e3)
print('Opex: ', sum(pv_operational_costs)/1e3)
print('NPV: ',sum(total_pv)/1e3)

plt.figure(figsize=(10, 3))
plt.plot(years, total_pv, color=teal2, label='Total costs')
plt.plot(years, pv_operational_costs, color=brown2, label='Opex')
plt.plot(years, pv_capital_costs, color=beige3, label='Capital Costs')
plt.xticks(years, [int(year) for year in years])
plt.xlabel('Year')
plt.ylabel('Present Value [Million €]')
plt.xlim(first_year,final_year)
plt.ylim(0,500)
plt.grid(axis='y')
plt.legend()
plt.show()

if save_to_csv:
    total_pv_df = pd.DataFrame(total_pv)
    try:
        df = pd.read_csv('../result_data/total_costs.csv')
    except FileNotFoundError:
        df = pd.DataFrame()
    df[case_path] = total_pv_df.iloc[:]
    df.to_csv('../result_data/total_costs.csv', index=False)

In [ ]:
# SUBSIDIES CALCULATION

subsidies = get_subsidies()
cost_of_subsidies = {}

for year in years:
    power_prod = get_power_prod(year)
    
    for (subsidy_year, carrier), subsidy_rate in subsidies.items():
        if subsidy_year != year:
            continue
        
        carrier_name = 'Oil' if carrier == 'oil' else carrier

        if carrier_name not in power_prod.columns or power_prod.empty:
            continue

        if year not in cost_of_subsidies:
            cost_of_subsidies[year] = {}
        if carrier_name not in cost_of_subsidies[year]:
            cost_of_subsidies[year][carrier_name] = 0

        prod_value = power_prod[carrier_name].iloc[0]

        cost_of_subsidies[year][carrier_name] += subsidy_rate * prod_value

cost_of_subsidies_df = pd.DataFrame()
cost_of_subsidies_df = pd.DataFrame.from_dict(cost_of_subsidies, orient='index')
cost_of_subsidies_df = rename_columns(cost_of_subsidies_df)

subsidies_list = list(cost_of_subsidies_df.sum(axis=1))
pv_subsidies = [calculate_present_value(i, year, years[0], discountrate) for i, year in zip(subsidies_list, years)]

npv_subsidies = sum(pv_subsidies)
print('NPV subsidies: ', npv_subsidies)
print(cost_of_subsidies_df.sum())
print('Total: ', cost_of_subsidies_df.sum().sum())

if save_to_csv:
    try:
        # Try to read the existing file
        df = pd.read_csv('../result_data/subsidies.csv')
    except FileNotFoundError:
        # If file does not exist, create a new DataFrame
        df = pd.DataFrame()

    # Add the new data as a column
    df[case_path] = [npv_subsidies]

    # Write the updated DataFrame to the file
    df.to_csv('../result_data/subsidies.csv', index=False)

In [ ]:
# Plot NPV and subsidies

import matplotlib.colors as mcolors
fig, ax = plt.subplots(figsize=(4, 4))  # Adjusted width for single bar

total_costs = [sum(pv_capital_costs)/1e3 + sum(pv_operational_costs)/1e3]
color_single = beige2

capex_bars = ax.bar(case, total_costs, label='NPV', color=color_single, edgecolor=color_single)

fill_colors_with_alpha = [mcolors.to_rgba(color_single, alpha=0.4)]
solid_edge_colors = [mcolors.to_rgba(color_single, alpha=1)]

subsidies_total = [sum(pv_subsidies)/1e3]

subsidies_bars = ax.bar(case, subsidies_total, label='OPEX', bottom=total_costs, 
                   color=fill_colors_with_alpha, edgecolor=solid_edge_colors, linestyle='--', linewidth=2)

ax.set_xticks(case)

# Legend
npv_patch = mpatches.Patch(color='darkgrey', label='NPV')
subsidies_patch = mpatches.Patch(facecolor=grey1, edgecolor='darkgrey', linestyle='--', linewidth=1, label='Subsidies')

plt.legend(handles=[subsidies_patch, npv_patch], loc='upper center', edgecolor=grey1, bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=14)
plt.ylabel('Cost [billion €]', fontsize=18)
plt.ylim(0, 8)

plt.grid(axis='y', color='grey')
plt.tight_layout()
plt.show()

print('Total NPV: ', sum(pv_capital_costs)/1e3 + sum(pv_operational_costs)/1e3)
print('Total subsidies: ', sum(pv_subsidies)/1e3)

### Emission plot

In [ ]:
# PLOT EMISSIONS

total_emissions = [get_emissions(year) for year in years]
if save_to_csv:
    total_emissions_df = pd.DataFrame({'Year': years, 'Emissions': total_emissions})

    try:
        # Try to read the existing file
        df = pd.read_csv('../result_data/emissions.csv')
    except FileNotFoundError:
        # If file does not exist, create a new DataFrame
        df = pd.DataFrame()
  


    if case_path not in df.columns:
        df[case_path] = total_emissions_df['Emissions']
    df.to_csv('../result_data/emissions.csv', index=False)


plt.figure(figsize=(10, 3))
plt.plot(years, total_emissions, color=teal3)
plt.ylabel('Total Emissions [Mt]')
plt.xlabel('Year')
plt.grid(axis='y')
plt.plot()

### Capacity and generation plots

In [ ]:
# INSTALLED CAPACITY FOR FIRST YEAR

installed_capacity = get_installed_capacity(first_year)
installed_capacity.index = [first_year]
installed_capacity = rename_columns(installed_capacity)
#installed_capacity_24.plot(kind='bar',color=get_colors(installed_capacity_24.columns),figsize=(10, 4), legend=True)
if save_to_csv:
    installed_capacity.to_csv('../result_data/' + case_path + '_24' + '_final_capacity.csv')
installed_capacity.plot(kind='bar',color=get_colors(installed_capacity.columns),figsize=(10, 4), legend=True)

In [ ]:
# INSTALLED CAPACITY FOR FINAL YEAR

installed_capacity = get_installed_capacity(final_year)
installed_capacity.index = [final_year]
installed_capacity = rename_columns(installed_capacity)
if save_to_csv:
    installed_capacity.to_csv('../result_data/' + case_path + '_final_capacity.csv')
installed_capacity.plot(kind='bar',color=get_colors(installed_capacity.columns),figsize=(10, 4), legend=True)

In [ ]:
# PLOT NEW INSTALLED CAPACITY PER YEAR

grouped_cap_change_B = get_new_installed(years, merge_ror=True)
grouped_cap_change_B = rename_columns(grouped_cap_change_B)
if save_to_csv:
    grouped_cap_change_B.to_csv('../result_data/' + case_path + '_new_capacity.csv')
grouped_cap_change_B.plot.bar(stacked=True, figsize=(12, 6),color=get_colors(grouped_cap_change_B.columns), legend=True)

print(grouped_cap_change_B.sum())

plt.ylabel('Additional Installed capacity [MW]')
plt.xlabel('')
plt.xticks(range(len(grouped_cap_change_B.index)), grouped_cap_change_B.index, rotation=90)
plt.title(case)
plt.ylim(0, 600)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(axis = 'y',zorder = 0)
#plt.savefig(path_fig + 'new_capacity.png', dpi=300, bbox_inches='tight')

In [ ]:
# PLOT NEW INSTALLED BATTERY CAPACITY PER YEAR

battery_change = get_new_installed_battery(years)
if save_to_csv:
    battery_change.to_csv('../result_data/' + case_path + '_new_battery_capacity.csv')
#battery_change = rename_columns(battery_change)
battery_change.plot.bar(stacked=True, figsize=(10, 6),color=[teal2, teal4, teal3])

plt.ylabel('Additional Installed capacity [MW]')
plt.xlabel('')
plt.xticks(range(len(battery_change.index)), battery_change.index, rotation=90)
plt.title(case)
plt.ylim(0, 5400)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(axis = 'y',zorder = 0)
#plt.savefig(path_fig + 'new_battery_capacity.png', dpi=300, bbox_inches='tight')

In [ ]:
# PLOT POWER PRODUCTION PER YEAR

production_sources = [get_power_prod(year) for year in years]
production = pd.concat(production_sources)
production.index = years
production = production.astype('float32')

if 'load' in production:
  production.drop('load', axis=1, inplace=True)

#production['hydro'] += production.pop('ror')
#print(production)
production = rename_columns(production)
if save_to_csv:
  production.to_csv('../result_data/' + case_path + '_production.csv')
production.plot.area(stacked=True, color=get_colors(production.columns),figsize=(12, 5))

plt.xticks(years, [int(year) for year in years])
plt.ylabel('Power Generation in TWh')
plt.xlabel('Year')
plt.xlim(first_year,final_year)
plt.ylim(0,25)
plt.title(case)
#plt.xticks(range(len(years)), years)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
#plt.savefig(path_fig + 'power_generation.png', dpi=300, bbox_inches='tight')
print(production.sum())

In [ ]:
# PLOT CAPACITY FACTORS

cp = pd.DataFrame({y: capacity_factors(y) for y in years}).T

if 'hydro_store' in cp.columns:
    cp['hydro'] = cp[['hydro','hydro_store']].mean(axis=1)
    cp = cp.drop(columns='hydro_store')

cp = rename_columns(cp)

cap = pd.concat([get_installed_capacity(y) for y in cp.index])
cap = rename_columns(cap)

cp = cp.mask(cap.reindex(columns=cp.columns).fillna(0) < 1e-3, 0)

fig, ax = plt.subplots(figsize=(8, 4))
cp.plot(marker='o', color=get_colors(cp.columns), ax=ax)

ax.set_xlabel('Year')
ax.set_ylabel('Capacity Factor')
ax.grid(axis='y')
ax.set_ylim(0, 1)
ax.set_xlim(first_year, final_year)
ax.set_yticks([0.00, 0.25, 0.50, 0.75, 1.00])
ax.legend().set_visible(False)

print(cp)


if save_to_csv:
    cp.to_csv('../result_data/' + case_path + '_capacity_factors.csv')

In [ ]:
# POWER MIX FINAL YEAR

power_mix_df = get_power_mix(final_year)
print(power_mix_df)
plt.rcParams['font.size'] = 10
power_mix_df = rename_columns(power_mix_df)
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(power_mix_df.iloc[0].values, autopct=custom_autopct, colors=get_colors(power_mix_df.columns), startangle=90, counterclock=False, wedgeprops=dict(width=0.3), pctdistance=1.1)

# Hide the zero-value autotexts
for autotext in autotexts:
    if autotext.get_text() <= '0.3%':
        autotext.set_visible(False)

ax.axis('equal')
centre_circle = plt.Circle((0,0),0.70,fc='white')
fig.gca().add_artist(centre_circle)

if save_to_csv:
    power_mix_df.to_csv('../result_data/' + case_path + '_power_mix.csv')

In [ ]:
# PLOT POWER MIX FOR FIRST AND FINAL YEAR

power_mix_first = get_power_mix(first_year)
power_mix_final = get_power_mix(final_year)
print(power_mix_final)

plt.rcParams['font.size'] = 14
power_mix_first = rename_columns(power_mix_first)
power_mix_final = rename_columns(power_mix_final)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

wedges1, texts1, autotexts1 = ax1.pie(
    power_mix_first.iloc[0].values, autopct=custom_autopct,
    colors=get_colors(power_mix_first.columns), startangle=90, counterclock=False, 
    wedgeprops=dict(width=0.3), pctdistance=1.1
)

wedges2, texts2, autotexts2 = ax2.pie(
    power_mix_final.iloc[0].values, autopct=custom_autopct,
    colors=get_colors(power_mix_final.columns), startangle=90, counterclock=False, 
    wedgeprops=dict(width=0.3), pctdistance=1.1
)

for autotext in autotexts1 + autotexts2:
    if autotext.get_text() <= '0.3%':
        autotext.set_visible(False)

centre_circle1 = plt.Circle((0,0),0.70,fc='white')
ax1.add_artist(centre_circle1)
ax1.set_title(str(first_year) +' Power Mix')

centre_circle2 = plt.Circle((0,0),0.70,fc='white')
ax2.add_artist(centre_circle2)
ax2.set_title(str(final_year) +' Power Mix')

ax1.axis('equal')
ax2.axis('equal')

fig.legend(power_mix_first.columns, loc="center right", bbox_to_anchor=(1, 0.5))
plt.tight_layout(pad=3.0)
fig.suptitle(case, fontsize=25)
#plt.savefig(path_fig + 'power_mix.png', dpi=300, bbox_inches='tight')

In [ ]:
# PLOT DISPATCH

first_date = "2013-12-28"
second_date = "2013-12-31"
year = final_year

CCGT = get_snapshot_generation(year, first_date, second_date, 'CCGT')
OCGT = get_snapshot_generation(year, first_date, second_date, 'OCGT')
Oil = get_snapshot_generation(year, first_date, second_date, 'oil')
Geothermal = get_snapshot_generation(year, first_date, second_date, 'geothermal')
Hydro = get_snapshot_generation(year, first_date, second_date, 'ror')
Hydro += get_snapshot_generation(year, first_date, second_date, 'hydro')
Wind = get_snapshot_generation(year, first_date, second_date, 'onwind')
Solar = get_snapshot_generation(year, first_date, second_date, 'solar')
Biomass = get_snapshot_generation(year, first_date, second_date, 'biomass')
Battery = get_snapshot_generation(year, first_date, second_date, 'battery')
demand = get_snapshot_demand(year, first_date, second_date)   

i=0
nbattery = []
pbattery =[]
for i in Battery:
    if i < 0:
        nbattery.append(i)
    else:
        nbattery.append(0) 
for i in Battery:
    if i > 0:
        pbattery.append(i)
    else:
        pbattery.append(0) 
LL = network.generators_t.p.loc[first_date:second_date].groupby(network.generators.carrier, axis=1).sum()['load']/1e3

snapshots = {'CCGT': CCGT, 'OCGT': OCGT, 'Oil': Oil,'Geothermal': Geothermal, 'Hydro': Hydro, 'Wind': Wind, 'Solar': Solar, 'Biomass': Biomass, 'Battery': pbattery, 'Nbattery': nbattery, 'Lost load': LL,'Demand': demand}
df_snapshots = pd.DataFrame(snapshots)
if save_to_csv:
    df_snapshots.to_csv('../result_data/' + case_path + '_snapshots.csv')

print('CCGT',CCGT.sum())
print('OCGT',OCGT.sum())
print('Oil',Oil.sum())
print('Geothermal',Geothermal.sum())
print('Hydro',Hydro.sum())
print('Wind',Wind.sum())
print('Solar',Solar.sum())
print('Biomass',Biomass.sum())
print('LL',LL.sum())
print('Demand', demand.sum())

print('Battery: ', Battery.sum())

fig, ax=plt.subplots(figsize = (12,6))
A = plt.stackplot(CCGT.index,CCGT,OCGT,Oil, Geothermal, Hydro, Wind, Solar, Biomass, pbattery, LL,
                  colors=[beige2,beige3,beige1,brown2, teal2, teal4,yellow,teal3, brown, grey4], zorder = 2) #,   purple3
plt.stackplot(CCGT.index,demand,nbattery, colors=[grey1, brown], zorder = 2) #,nbattery
plt.xticks(rotation = 0)
plt.yticks()
scale_y = 1e3
ticks_y = ticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x/scale_y))
ax.yaxis.set_major_formatter(ticks_y)
myFmt = mdates.DateFormatter('%d %b')
ax.xaxis.set_major_formatter(myFmt)
ax.set_ylabel('Generation [GW]')

start_date = pd.to_datetime(first_date)
end_date = pd.to_datetime(second_date)

ax.set_xlim(start_date, end_date)
ax.set_ylim(-4000, 4000)


biomass_patch = mpatches.Patch(color = teal3, label = 'Biomass')
solar_patch = mpatches.Patch(color=yellow, label ='Solar')
wind_patch = mpatches.Patch(color=teal4, label ='Wind')
hydro_patch = mpatches.Patch(color = teal2, label = 'Hydro')
geo_patch = mpatches.Patch(color = brown2, label = 'Geothermal')
oil_patch = mpatches.Patch(color=beige1, label = 'Oil')
OCGT_patch = mpatches.Patch(color=beige3, label ='OCGT')
CCGT_patch = mpatches.Patch(color=beige2, label ='CCGT')
load_patch = mpatches.Patch(color=grey1, label ='Demand')
battery_patch = mpatches.Patch(color=brown, label ='Battery')
LL_patch = mpatches.Patch(color=grey4, label ='Lost Load')
#H2_patch = mpatches.Patch(color=red2, label ='H2')
handles=[biomass_patch, solar_patch, wind_patch, hydro_patch, geo_patch,oil_patch, OCGT_patch, CCGT_patch, load_patch, battery_patch, LL_patch]
ax.legend(handles=handles, frameon = False, loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(axis='y', zorder=0)
plt.title(case + ' - 2040')
#plt.savefig(path_fig + 'snapshot.png', dpi=300, bbox_inches='tight')